# Spectrum Objects

`specreduce` does not invent its own data class. Every operation in the package consumes and
returns a `specutils` **`Spectrum`**, and the previous chapter's `parse_image` exists to turn
whatever you hand in into one. Knowing what that container holds, and just as importantly what
it quietly fills in when you leave something out, is the last piece of groundwork before the
reduction starts.

A `Spectrum` bundles four things that belong together:

| part | what it is | why it matters |
|---|---|---|
| `flux` | the data, as an `astropy` `Quantity` | carries its unit, so `electron` and `DN` cannot be confused |
| `spectral_axis` | one coordinate per sample | pixels before calibration, wavelength after |
| `uncertainty` | per-sample variance or standard deviation | drives extraction weights, clipping, detection thresholds |
| `mask` | boolean, `True` where data is bad | keeps dead pixels out of fits |

In [ ]:
import sys

sys.path.append("../src")

import astropy.units as u
import numpy as np
from astropy.nddata import (
    InverseVariance,
    StdDevUncertainty,
    VarianceUncertainty,
)
from specreduce.core import parse_image
from specutils import Spectrum

from common import read_data

arcs, lamps, obj = read_data()

## Building one by hand

The constructor takes the four pieces directly. Nothing here is specific to `specreduce`.

In [ ]:
counts = np.array([12.0, 40.0, 95.0, 41.0, 11.0])

demo = Spectrum(
    flux=counts * u.electron,
    spectral_axis=np.arange(counts.size) * u.pix,
    uncertainty=VarianceUncertainty(counts + 13.0**2),
    mask=np.array([False, False, False, True, False]),
)

print(f"flux           {demo.flux}")
print(f"spectral_axis  {demo.spectral_axis}")
print(f"uncertainty    {demo.uncertainty.uncertainty_type}, {demo.uncertainty.array}")
print(f"mask           {demo.mask}")
print(f"shape          {demo.shape}")

Note the uncertainty is stored as a **variance**, not a standard deviation, and the object records
which convention it uses in `uncertainty_type`. That distinction is the source of more friction
than it deserves, and we come back to it below.

## A `Spectrum` is not necessarily one-dimensional

The name is misleading. `Spectrum` is the container for anything with a spectral axis, and that
includes the whole 2D frame. `parse_image`, the function every `specreduce` operation calls on its
input, returns a **2D** `Spectrum`:

In [ ]:
frame = parse_image(obj)

print(f"flux shape           {frame.flux.shape}")
print(f"spectral_axis shape  {frame.spectral_axis.shape}")
print(f"spectral_axis_index  {frame.spectral_axis_index}")
print(f"uncertainty shape    {frame.uncertainty.array.shape}")

`spectral_axis_index` is the attribute that makes sense of this: it says which array axis the
spectral axis describes. Here it is 1, matching Chapter 2's convention: dispersion on axis 1,
cross-dispersion on axis 0. The spectral axis itself has 1000 entries, one per dispersion column,
and is shared by all 372 rows.

So the same class covers the 2D frame going in and the 1D spectrum coming out. When reading
`specreduce` signatures, "image" and "spectrum" are both `Spectrum` objects; the difference is
dimensionality.

## Uncertainties: three conventions, one quantity

`astropy.nddata` offers three ways to express the same information, and they are not
interchangeable in code even though they are in meaning.

In [ ]:
variance = VarianceUncertainty([4.0, 9.0, 25.0])

print(f"{'class':>22}  {'uncertainty_type':>17}  {'array':>22}")
for converted in (
    variance,
    variance.represent_as(StdDevUncertainty),
    variance.represent_as(InverseVariance),
):
    print(
        f"{type(converted).__name__:>22}  {converted.uncertainty_type:>17}  "
        f"{str(converted.array):>22}"
    )

`represent_as` converts between them. The book's own convention is `VarianceUncertainty`, which is
what `common.read_file` attaches, what `Background` propagates, and what both extraction classes
return. Variances add linearly, which makes propagation arithmetic simple.

`specreduce` accepts any of the three. `find_arc_lines` (Chapter 9), for instance, works in
standard deviations internally (its `noise_factor` threshold is a multiple of the uncertainty), so
it converts whatever it is given, on a copy, and leaves the input untouched. You will still meet
`represent_as` when *you* need a standard deviation, as the error bars in a plot or the
`sigma` of a fit, and when a `specutils` routine wants one type in particular.

The uncertainty is not decoration. It sets extraction weights in `HorneExtract`, the clipping scale
in `Background`, and the detection threshold in `find_arc_lines`. Chapter 2 showed that a bare
`ndarray` acquires a variance of exactly 1.0, which is why the book always loads through
`common.read_data`.

## Units

Two independent unit systems live in a `Spectrum`: one on the flux, one on the spectral axis.

The **spectral axis** converts freely, because wavelength, frequency and energy are all the same
physical quantity viewed differently.

In [ ]:
calibrated_axis = Spectrum(
    flux=counts * u.electron,
    spectral_axis=(5000 + np.arange(counts.size) * 5.0) * u.AA,
    uncertainty=VarianceUncertainty(counts + 13.0**2),
)

print(f"wavelength  {calibrated_axis.spectral_axis}")
print(f"frequency   {calibrated_axis.spectral_axis.to(u.THz).round(3)}")
print(
    f"energy      {calibrated_axis.spectral_axis.to(u.eV, equivalencies=u.spectral()).round(4)}"
)

The **flux** unit does not convert so freely, and that is a feature:

In [ ]:
try:
    calibrated_axis.with_flux_unit(u.ct)
except u.UnitConversionError as error:
    print(f"UnitConversionError: {error}")

scaled = calibrated_axis.with_flux_unit(u.Unit(1e3 * u.electron))
print(f"\nrescaling the same unit is fine: {scaled.flux}")

`electron` and `ct` (count / ADU) are *not* convertible without knowing the detector gain, and
`astropy` refuses to guess. This is the units system doing its job: Chapter 1 multiplied by `GAIN`
precisely because photon statistics apply to detected charge, not to digitized counts, and a
container that let you silently relabel one as the other would make that distinction meaningless.

## The spectral axis and the WCS

The `spectral_axis` is the coordinate of each sample along the dispersion direction. It is a
`SpectralAxis`, a `Quantity` subclass, with one entry per sample: a pixel index before wavelength
calibration, a wavelength (or frequency, or energy) after. On a 2D frame there is still only one
axis, shared by every row.

The `wcs` holds the same information as a *transform* rather than a table of values: a mapping from
array index to world coordinate that can be evaluated anywhere, not only at the sample centers.
`Spectrum` accepts either. Given a `spectral_axis`, `specutils` builds a lookup-table WCS (a
`gwcs` object) from its values; given a `wcs`, it evaluates the transform at each pixel to fill in
the axis. The two stay consistent, so `spectral_axis` is always present and is the attribute to
read in ordinary code. The `wcs` is what a file format stores when a spectrum with a wavelength
solution is written out, and what a reader evaluates to recover the axis.

On a pixel axis the world coordinate has no physical meaning, and `astropy` will not build a
`SpectralCoord` from it, so before calibration only the `spectral_axis` is usable.

## Summary

- Every `specreduce` operation speaks `specutils` **`Spectrum`**: `flux` (a `Quantity`),
  `spectral_axis`, `uncertainty`, and `mask` in one object.
- **A `Spectrum` can be 2D.** `parse_image` returns the whole frame as one, with
  `spectral_axis_index` saying which axis is spectral (1, per Chapter 2's convention).
- Uncertainties come in three interchangeable conventions, and `specreduce` accepts all of them.
  The book uses `VarianceUncertainty`; `represent_as` converts when you need another.
- The spectral axis converts freely between wavelength, frequency and energy. The flux unit does
  not: `electron` and `ct` need the gain, and `astropy` will not guess.
- The `spectral_axis` is one coordinate per sample, a pixel index before calibration and a
  wavelength after. The `wcs` is the same mapping as a transform; `specutils` builds whichever
  one you did not supply, so `spectral_axis` is always there to read.

That completes the foundations. Next: [Tracing the Spectrum](04-tracing.ipynb), where the reduction
proper begins.